# Perks Assignment

In this notebook we'll assign the most appropriate perks to the segmented users.

## Bootstrap
Configuring parameters, loading libraries and dataset.

In [71]:
import sys, os
# This line allows us to import from the parent directory, which is where the 'src' folder is located.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
# These lines enable automatic reloading of modules in Jupyter, so that changes to the code are reflected without needing to restart the kernel.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Libraries
External

In [72]:
import pandas as pd
import seaborn as sns
import numpy as np

Project libraries

In [73]:
from src.utils import Config

### Configuration
Set here general paramenters

In [74]:
config = Config({
    'datasets_paths': {
        'original': '../data/aggregated/users.csv',
        'machine_learning': '../data/segmented/users_ml.csv',
        'rule_based': '../data/segmented/users_rb.csv',
    },
    'save_perks_to_path': '../data/assigned/perks.csv',
    'save_assignment_to_path': '../data/assigned/users.csv',
})

### Loading Datasets

In [75]:
users = pd.read_csv(config.datasets_paths.original)
users_ml = pd.read_csv(config.datasets_paths.machine_learning)
users_rb = pd.read_csv(config.datasets_paths.rule_based)

## Perks Definition
This is the list of _perks_ TravelTide can offer to its customers.

In [76]:
perks = {
    'free_hotel_night': {
        'description': 'One free hotel night for every 10 nights booked',
        'eligibility_criteria': 'Users in the "Frequent Travelers" segment',
        'benefits': 'Encourages repeat bookings and rewards loyal customers',
    },
    'priority_boarding': {
        'description': 'Priority boarding for flights',
        'eligibility_criteria': 'Users in the "Frequent Travelers", "Business Travelers", and "High Spenders" segments',
        'benefits': 'Enhances the travel experience for frequent and high-value customers, making them feel valued and appreciated',
    },
    'free_breakfast': {
        'description': 'Free breakfast at partner hotels',
        'eligibility_criteria': 'Users in the "Business Travelers" and "High Spenders" segments',
        'benefits': 'Adds value to hotel bookings for business travelers and high spenders, improving customer satisfaction and loyalty',
    },
    'free_checked_bag': {
        'description': 'One free checked bag for flights',
        'eligibility_criteria': 'Users in the "Frequent Travelers" segment',
        'benefits': 'Provides a tangible benefit for frequent travelers, encouraging them to continue booking with the platform',
    },
    'no_cancellation_fee': {
        'description': 'No cancellation fee for bookings',
        'eligibility_criteria': 'Users in the "Business Travelers" segment',
        'benefits': 'Offers flexibility for business travelers who may have changing plans, increasing their likelihood of booking through the platform',
    },
    'free_seat_selection': {
        'description': 'Free seat selection for flights',
        'eligibility_criteria': 'Users in the "High Spenders" segment',
        'benefits': 'Provides an added perk for high-value customers, enhancing their booking experience and encouraging loyalty',
    },
    'more_miles_earned': {
        'description': 'Earn 2x miles on all bookings',
        'eligibility_criteria': 'Users in the "Frequent Travelers" segment',
        'benefits': 'Incentivizes frequent travelers to book more through the platform, as they can earn more rewards for their loyalty',
    },
    'discount_flight': {
        'description': '10% discount on flight bookings',
        'eligibility_criteria': 'Users in the "Frequent Travelers" segment',
        'benefits': 'Provides a financial incentive for frequent travelers to choose the platform for their flight bookings, increasing customer retention and satisfaction',
    },
    'discount_hotel': {
        'description': '10% discount on hotel bookings',
        'eligibility_criteria': 'Users in the "Business Travelers" segment',
        'benefits': 'Offers a cost-saving benefit for business travelers, encouraging them to book more hotels through the platform and increasing customer loyalty',
    },
    'free_lounge_access': {
        'description': 'Free airport lounge access',
        'eligibility_criteria': 'Users in the "Business Travelers" segment',
        'benefits': 'Enhances the travel experience for business travelers and adds value to their bookings',
    },
    'priority_support': {
        'description': 'Priority customer support',
        'eligibility_criteria': 'Users in the "High Spenders" segment',
        'benefits': 'Provides personalized assistance to high-value customers, improving satisfaction and retention',
    },
}

## Perk Asignment
In this section we assign a perk to each segment.

In [77]:
# Define the logic for perk assignment
perk_map = {
    # High Frequency & Business-ish
    'independent_frequent_travelers': 'priority_boarding',
    'frequent_travelers': 'more_miles_earned',
    'business_travelers': 'free_lounge_access',
    
    # Mature & Seniors
    'silver_adventurers': 'priority_support',
    'family_oriented_mature_couples': 'free_hotel_night',
    'senior_premium_hotel_only_couples': 'free_breakfast',
    'inactive_senior_couples': 'no_cancellation_fee',
    
    # Families & Couples
    'family_vacationers': 'free_checked_bag',
    'inactive_young_parents': 'discount_flight',
    'married_couples': 'free_seat_selection',
    
    # Specific Booking Behaviors
    'independent_bundle_oriented_travelers': 'discount_hotel',
    'young_premium_hotel_only_travelers': 'free_breakfast',
    'flight_only_users': 'discount_hotel', # Upsell logic
    'last_minute_bookers': 'no_cancellation_fee',
    
    # Budget & Re-engagement
    'price_sensitive_travelers': 'discount_flight',
    'inactive_young_non_travelers': 'discount_flight',
    'high_value_customers': 'free_lounge_access',
    'leisure_travelers': 'free_checked_bag'
}

In [78]:
# Saving perks
for segment, perk in perk_map.items():
    perks[perk]['assignments'] = perks[perk].get('assignments', []) + [segment]
perks_df = pd.DataFrame.from_dict(perks, orient='index').reset_index().rename(columns={'index': 'perk_id'})
perks_df.to_csv(config.save_perks_to_path, index=False)

In [80]:
users['segment'] = np.nan
users['perk'] = np.nan
for user_id, segment in users_rb[['user_id', 'segment']].itertuples(index=False):
    if pd.isna(segment): continue
    segments = segment.split(',')  # Handle multiple segments
    perk = np.nan
    for seg in segments:
        if seg in perk_map:
            perk = perk_map[seg]
            break
    users.loc[users.user_id == user_id, 'perk'] = perk
    users.loc[users.user_id == user_id, 'segment'] = segment
for user_id, segment in users_ml[['user_id', 'segment']].itertuples(index=False):
    if pd.isna(segment): continue
    perk = perk_map[segment] if segment in perk_map else np.nan
    users.loc[users.user_id == user_id, 'perk'] = perk
    users.loc[users.user_id == user_id, 'segment'] = segment

users.to_csv(config.save_assignment_to_path, index=False)

/var/folders/_7/tz1m5gcs1nlff79vj4knvqz00000gn/T/ipykernel_69131/2342084967.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'free_checked_bag' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  users.loc[users.user_id == user_id, 'perk'] = perk
/var/folders/_7/tz1m5gcs1nlff79vj4knvqz00000gn/T/ipykernel_69131/2342084967.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'leisure_travelers' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  users.loc[users.user_id == user_id, 'segment'] = segment
